# Face-depth difference features

This notebook runs the recovered difference-feature calculation on a synthetic
depth plane. Change the plane slopes and rerun the cells to inspect the feature
matrix. It does not predict depth from a color image: the classifier, calibration,
training data and getDepthFeatures helper remain unavailable. No face photographs
or learned model are included.


Updated component for Aaron Wetzler, 2026-11-12. Helper/tests drafted with Claude Code; reviewed and completed with Codex. Original student attribution and source remain in source/original/.


In [ ]:
from pathlib import Path
import sys

module_name = "face_depth_features"
candidates = [Path.cwd(), *Path.cwd().parents]
local = next((p / "updated" for p in candidates
              if (p / "updated" / (module_name + ".py")).is_file()), None)
if local is None:
    # Colab: download only the reviewed helper, not any data or legacy service.
    from urllib.request import urlopen
    target = Path(module_name + ".py")
    source_url = "https://raw.githubusercontent.com/twerdster/face-depth-estimation-from-a-single-color-image/main/updated/" + target.name
    target.write_bytes(urlopen(source_url, timeout=30).read())
    local = Path.cwd()
sys.path.insert(0, str(local))


In [ ]:
from face_depth_features import get_diff, probe_radius_profile

# Change the parameters and rerun.
slope_row, slope_col, offset = 3, 2, 50
height, width = 25, 39
D = [[offset + slope_row*r + slope_col*c for c in range(width)]
     for r in range(height)]
CM = [[1 for c in range(width)] for r in range(height)]
features, labels = get_diff(D, CM)
center_row, center_col = height//2, width//2
# For this positive plane, MATLAB find order visits every pixel column-first.
pixel_index = center_col * height + center_row
patch = features[21*pixel_index:21*(pixel_index+1)]
assert len(features) == height * width * 21
assert patch[10][17] == 0
print("Feature matrix:", len(features), "x", len(features[0]))
print("Each center label is repeated across its 21 patch rows.")
print("Probe-radius endpoints:", probe_radius_profile()[0], probe_radius_profile()[-1])


In [ ]:
from IPython.display import SVG, display

scale = max(1, max(abs(v) for row in patch for v in row))
tiles = []
for r, row in enumerate(patch):
    for c, v in enumerate(row):
        shade = round(127 + 120*v/scale)
        tiles.append(f'<rect x="{c*12}" y="{r*12}" width="12" height="12" '
                     f'fill="rgb({shade},{shade},{shade})"/>')
display(SVG('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 420 252" '
            'role="img" aria-label="Synthetic depth difference patch, dark negative and light positive">'
            + "".join(tiles) + '</svg>'))
print("Patch range:", min(map(min, patch)), "to", max(map(max, patch)))
print("Values are synthetic depth differences; this is not a reconstructed face.")


## Scope and differences

The historical 21-by-35 patches are stacked vertically, not flattened into
one row per pixel. Nonzero pixels follow MATLAB column-major find order. Outside
the image the feature is zero; an in-image zero depth gives zero minus the center
depth. build_diff replaces NaNs before feature construction, as the original
script does. The port adds rectangular-input and positive-integer extent checks.
The MATLAB classifier training, its 502-frame loop and random probe shuffle are
not ported.
